In [12]:
# Load data
import pandas as pd, numpy as np, json
from dowhy import CausalModel
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/flipkart_processed.csv')

# For causal analysis, simulate quantity sold from price elasticity
# Real Kaggle dataset has no sales quantity — standard practice to simulate
# Elasticity: qty decreases as price increases vs competitors
np.random.seed(42)
BASE_QTY = 50
df['qty'] = (
    BASE_QTY
    - 0.3 * (df['discounted_price'] - df['comp_avg_price']) / df['comp_avg_price'] * BASE_QTY
    + np.random.normal(0, 5, len(df))
).clip(1, 500).round().astype(int)

print(f"Simulated qty | mean: {df['qty'].mean():.1f} | std: {df['qty'].std():.1f}")
print(df[['discounted_price','comp_avg_price','qty']].describe())

Simulated qty | mean: 50.0 | std: 5.1
       discounted_price  comp_avg_price           qty
count      18269.000000    18269.000000  18269.000000
mean        1524.545514     1514.624792     50.003667
std         4185.841567     4156.509499      5.061667
min          120.000000      120.000000      1.000000
25%          350.000000      350.000000     47.000000
50%          549.000000      549.000000     50.000000
75%          999.000000      999.000000     53.000000
max        31916.000000    31795.333333     72.000000


In [13]:
# DoWhy causal graph
# Causal question: What is the effect of PRICE on QUANTITY SOLD?
# DAG: competitor_price → price → qty
#      discount_pct     → price
#      category         → price, qty

causal_df = df[['discounted_price', 'qty', 'comp_avg_price',
                 'discount_pct', 'category_encoded']].copy()
causal_df.columns = ['price', 'qty', 'comp_price', 'discount_pct', 'category']

# Sample for speed (DoWhy slow on 10k+ rows)
sample = causal_df.sample(min(3000, len(causal_df)), random_state=42)

model_causal = CausalModel(
    data=sample,
    treatment='price',
    outcome='qty',
    common_causes=['comp_price', 'discount_pct', 'category'],
    graph="""
    digraph {
        comp_price -> price;
        discount_pct -> price;
        category -> price;
        category -> qty;
        price -> qty;
    }
    """
)

identified = model_causal.identify_effect(proceed_when_unidentifiable=True)
print(identified)

Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
   d                     
────────(E[qty|category])
d[price]                 
Estimand assumption 1, Unconfoundedness: If U→{price} and U→qty then P(qty|price,category,U) = P(qty|price,category)

### Estimand : 2
Estimand name: iv
Estimand expression:
 ⎡                                                                       -1⎤
 ⎢             d                   ⎛             d                      ⎞  ⎥
E⎢───────────────────────────(qty)⋅⎜───────────────────────────([price])⎟  ⎥
 ⎣d[discount_pct  comp_price]      ⎝d[discount_pct  comp_price]         ⎠  ⎦
Estimand assumption 1, As-if-random: If U→→qty then ¬(U →→{discount_pct,comp_price})
Estimand assumption 2, Exclusion: If we remove {discount_pct,comp_price}→{price}, then ¬({discount_pct,comp_price}→qty)

### Estimand : 3
Estimand name: frontdoor
No such variable(s) found!

### Estimand : 4
Estimand name: general_adjustment
Esti

In [14]:
# Estimate effect
estimate = model_causal.estimate_effect(
    identified,
    method_name="backdoor.linear_regression",
    control_value=sample['price'].quantile(0.25),
    treatment_value=sample['price'].quantile(0.75),
)

elasticity = float(estimate.value)
print(f"Causal price elasticity: {elasticity:.6f}")
print(f"Interpretation: 1 unit ↑ price → {elasticity:.4f} change in qty")

Causal price elasticity: -0.020854
Interpretation: 1 unit ↑ price → -0.0209 change in qty


In [15]:
#  Compute optimal price
# Revenue-maximizing price given causal elasticity
# Revenue = price * qty = price * (base_qty + elasticity * (price - base_price))
# dR/dp = 0 → optimal_price = (base_qty - elasticity * base_price) / (-2 * elasticity)

base_price = float(sample['price'].mean())
base_qty   = float(sample['qty'].mean())

optimal_price = (base_qty - elasticity * base_price) / (-2 * elasticity + 1e-9)
optimal_price = np.clip(optimal_price, base_price * 0.5, base_price * 1.5)

print(f"Base price:    ₹{base_price:,.2f}")
print(f"Base qty:      {base_qty:.1f}")
print(f"Elasticity:    {elasticity:.6f}")
print(f"Optimal price: ₹{optimal_price:,.2f}")

Base price:    ₹1,522.63
Base qty:      50.0
Elasticity:    -0.020854
Optimal price: ₹1,961.05


In [16]:
# Save causal state
causal_state = {
    "elasticity":  round(elasticity, 8),
    "base_price":  round(base_price, 2),
    "base_qty":    round(base_qty, 2),
    "optimal_price": round(float(optimal_price), 2),
    "dataset":     "flipkart_kaggle",
}

with open('../models/causal_state.json', 'w') as f:
    json.dump(causal_state, f, indent=2)

print("✅ Causal state saved to models/causal_state.json")
print(causal_state)

✅ Causal state saved to models/causal_state.json
{'elasticity': -0.02085423, 'base_price': 1522.63, 'base_qty': 50.04, 'optimal_price': 1961.05, 'dataset': 'flipkart_kaggle'}
